# Synthetic Final Output Stage

This notebook builds the final Kafka/Postgres synthetic output so it matches the Kaggle-style dataset shape:

- `dataset_id`
- `run_id`
- `asset_id`
- `timestamp`
- `sensor_00` ... `sensor_51`
- `machine_status`

It uses the rebuilt stage as the source table and does **not** read premelt for this final output step.


In [5]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.synthetic_final_aligned_output_writer import build_synthetic_final_aligned_output_stage

In [6]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

NUMBER_OF_SENSORS = int(52)

IF_EXISTS_FLAG = str("replace")
COMPLETE_ONLY_FLAG = True
OBSERVATION_WINDOW_SIZE = int(2500)

REBUILT_SOURCE_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")
TARGET_TABLE_NAME = str("synthetic_sensor_observations_final_output")

TIMESTAMP_SOURCE_PRIORITY = (
    "observation_timestamp",
    "timestamp",
    "created_at",
)

STATUS_SOURCE_PRIORITY = (
    "machine_status",
    "stream_state",
    "phase",
)

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}


In [7]:
engine = get_engine_from_env()

In [ ]:
final_output_result = build_synthetic_final_aligned_output_stage(
    engine=engine,
    schema=SCHEMA,
    rebuilt_table=REBUILT_SOURCE_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    if_exists=IF_EXISTS_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
    timestamp_source_priority=TIMESTAMP_SOURCE_PRIORITY,
    status_source_priority=STATUS_SOURCE_PRIORITY,
    status_mapping=STATUS_MAPPING,
    timestamp_output_column="timestamp",
    machine_status_output_column="machine_status",
)


In [ ]:

final_output_result

----

---

## Validation

This confirms that the final table has the expected row count footprint and timestamp range.


In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT dataset_id) AS dataset_id_count,
        COUNT(DISTINCT run_id) AS run_id_count,
        COUNT(DISTINCT asset_id) AS asset_id_count,
        MIN(timestamp) AS min_timestamp,
        MAX(timestamp) AS max_timestamp,
        COUNT(*) FILTER (WHERE machine_status = 'NORMAL') AS normal_rows,
        COUNT(*) FILTER (WHERE machine_status = 'BROKEN') AS broken_rows,
        COUNT(*) FILTER (WHERE machine_status = 'RECOVERING') AS recovering_rows
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    """
)

display(validation_dataframe)


In [ ]:

display(validation_dataframe)

----

---

## Sample Output

This shows the final Kaggle-shaped output table layout.


In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        timestamp,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        machine_status
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    ORDER BY timestamp, dataset_id, run_id, asset_id
    LIMIT 10
    """
)

display(sample_dataframe)


In [ ]:
status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        machine_status,
        COUNT(*) AS row_count
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    GROUP BY machine_status
    ORDER BY machine_status
    """
)

display(status_distribution_dataframe)
